In [1]:
import os
import pandas as pd
from dotenv import load_dotenv

from qdrant_client import models, QdrantClient

In [2]:
# Carregar as variáveis do .env
load_dotenv()

qdrant_api = os.getenv("QDRANT_API_KEY")
qdrant_url = os.getenv("QDRANT_URL")
PATH = "/home/helder/Projects/crown/tcc-qamethod-snpg/data/data_treatment.csv"

In [3]:
df_qa_method = pd.read_csv(PATH)
df_qa_method.head()

,autor,titulo,tipo,area_de_concentracao,ano_de_publicacao,local,orientador,coorientador,resumo,palavras_chave,abstract,keywords,introducao,conclusao,embedding,x,y,z
0,jorge ivan hmeljevski,modelo para sistemas de supervisão de mercado ...,tese,engenharia do conhecimento,2021,florianópolis,jose leomar todesco,alexandre leopoldo goncalves,a confiança na higidez dos mercados de capitai...,"['mercado de capitais', 'mercado de valores mo...",the confidence in the integrity of capital mar...,"['capital market', 'securities market', 'marke...",{'contextualizacao': 'o mercado de capitais é ...,o modelo elaborado nest a pesquisa envolveu a ...,"[0.02262425608932972, 0.06360490620136261, 0.0...",-0.174124,0.159395,-0.060768
1,ivam galvão filho,fractus: aplicativo para aprendizagem de frações,dissertação,engenharia do conhecimento,2022,florianópolis,vania ribas ulbricht,elisa maria pivetta,o objetivo principal desta pesquisa foi o dese...,"['objetos de aprendizagem.', 'frações.', 'apli...",the main objective of this research was the de...,"['learning objects.', 'fractions.', 'app for l...",{'contextualizacao': 'o sistema educacional br...,o trabalho realizado pela organização todos pe...,"[-0.006245349999517202, 0.065089151263237, 0.0...",0.143998,0.263873,-0.055491
2,márcio crescencio,modelo de uma rede colaborativa suportada por ...,tese,engenharia do conhecimento,2022,florianópolis,alexandre augusto biz,jose leomar todesco,a convergência entre o turismo e a cultura atr...,"['sítios de patrimônio mundial', 'gestão do tu...",the convergence between tourism and culture th...,"['world heritage sites', 'tourism management',...",{'contextualizacao': 'o turismo se tornou uma ...,esta tese identificou que o turismo possui um ...,"[0.024945586919784546, 0.04501194879412651, 0....",-0.111669,-0.167909,-0.015695
3,roseli honorio,modelo conceitual de governança de dados como ...,dissertação,engenharia do conhecimento,2022,florianópolis,joao artur de souza,patricia de sa freire,a humanidade passou por transformações e revol...,"['governança de dados', 'governança do conheci...",humanity has undergone transformations and rev...,"['data governance', 'knowledge governance', 'f...",{'contextualizacao': 'a sociedade está atraves...,"na ind ústria 4.0 e sociedade 5.0, as organiza...","[0.05032069981098175, 0.051356106996536255, 0....",-0.229669,0.045873,-0.059402
4,josé tadeu silva,análise da contribuição da engenharia do conhe...,dissertação,engenharia do conhecimento,2022,florianópolis,fernando alvaro ostuni gauthier,marcelo macedo,a presente dissertação aborda as questões emer...,"['comércio eletrônico', 'modelo de análise', '...",the present dissertation addresses the emergin...,"['e-commerce', 'analysis model', 'knowledge en...",{'contextualizacao': 'de acordo com lemos (200...,a partir do objetivo geral de analisar as cont...,"[-0.0012473083334043622, 0.044789645820856094,...",-0.195246,0.141014,0.098894


In [4]:
import ast

df_qa_method['introducao_dict'] = df_qa_method['introducao'].apply(ast.literal_eval)

# Depois, crie as novas colunas extraindo os valores do dicionário
df_qa_method['introducao_contextualizacao'] = df_qa_method['introducao_dict'].apply(lambda x: x.get('contextualizacao', ''))
df_qa_method['introducao_problematica'] = df_qa_method['introducao_dict'].apply(lambda x: x.get('problematica', ''))
df_qa_method['introducao_ineditismo'] = df_qa_method['introducao_dict'].apply(lambda x: x.get('ineditismo', ''))
df_qa_method['introducao_contribuicao'] = df_qa_method['introducao_dict'].apply(lambda x: x.get('contribuição', ''))

# Remova a coluna temporária
df_qa_method.drop(columns=['introducao_dict'], inplace=True)

df_qa_method.head()

,autor,titulo,tipo,area_de_concentracao,ano_de_publicacao,local,orientador,coorientador,resumo,palavras_chave,...,introducao,conclusao,embedding,x,y,z,introducao_contextualizacao,introducao_problematica,introducao_ineditismo,introducao_contribuicao
0,jorge ivan hmeljevski,modelo para sistemas de supervisão de mercado ...,tese,engenharia do conhecimento,2021,florianópolis,jose leomar todesco,alexandre leopoldo goncalves,a confiança na higidez dos mercados de capitai...,"['mercado de capitais', 'mercado de valores mo...",...,{'contextualizacao': 'o mercado de capitais é ...,o modelo elaborado nest a pesquisa envolveu a ...,"[0.02262425608932972, 0.06360490620136261, 0.0...",-0.174124,0.159395,-0.060768,o mercado de capitais é fundamental para o cre...,"os ssm, de maneira geral, usam os dado s prove...","de maneira inédita, portanto, este trabalho pa...","a contribuição desta pesquisa, portanto, está ..."
1,ivam galvão filho,fractus: aplicativo para aprendizagem de frações,dissertação,engenharia do conhecimento,2022,florianópolis,vania ribas ulbricht,elisa maria pivetta,o objetivo principal desta pesquisa foi o dese...,"['objetos de aprendizagem.', 'frações.', 'apli...",...,{'contextualizacao': 'o sistema educacional br...,o trabalho realizado pela organização todos pe...,"[-0.006245349999517202, 0.065089151263237, 0.0...",0.143998,0.263873,-0.055491,o sistema educacional brasileiro passa por uma...,a deficiência na aprendizagem nas escolas de e...,,
2,márcio crescencio,modelo de uma rede colaborativa suportada por ...,tese,engenharia do conhecimento,2022,florianópolis,alexandre augusto biz,jose leomar todesco,a convergência entre o turismo e a cultura atr...,"['sítios de patrimônio mundial', 'gestão do tu...",...,{'contextualizacao': 'o turismo se tornou uma ...,esta tese identificou que o turismo possui um ...,"[0.024945586919784546, 0.04501194879412651, 0....",-0.111669,-0.167909,-0.015695,o turismo se tornou uma das maiores indústrias...,"a convergência entre o turismo e a cultura, at...",o reconhecimento de pm atrai turistas adiciona...,esses elementos de ligação do desenvolvimento ...
3,roseli honorio,modelo conceitual de governança de dados como ...,dissertação,engenharia do conhecimento,2022,florianópolis,joao artur de souza,patricia de sa freire,a humanidade passou por transformações e revol...,"['governança de dados', 'governança do conheci...",...,{'contextualizacao': 'a sociedade está atraves...,"na ind ústria 4.0 e sociedade 5.0, as organiza...","[0.05032069981098175, 0.051356106996536255, 0....",-0.229669,0.045873,-0.059402,a sociedade está atravessando um momento de mu...,é nesse contexto que a governanç a do conhecim...,,
4,josé tadeu silva,análise da contribuição da engenharia do conhe...,dissertação,engenharia do conhecimento,2022,florianópolis,fernando alvaro ostuni gauthier,marcelo macedo,a presente dissertação aborda as questões emer...,"['comércio eletrônico', 'modelo de análise', '...",...,{'contextualizacao': 'de acordo com lemos (200...,a partir do objetivo geral de analisar as cont...,"[-0.0012473083334043622, 0.044789645820856094,...",-0.195246,0.141014,0.098894,"de acordo com lemos (2003), sob qualquer aspec...","para vilaça e araújo (2016), o conhecimento so...",,


### Configuração do cliente do Qdrant

In [5]:
client_qdrant = QdrantClient(
    url=qdrant_url,
    api_key=qdrant_api,
)

In [6]:
index_name = "PPGEGC_UFSC"
dimensions = 1536


### Criação da collection

In [7]:
collection_exists = client_qdrant.collection_exists(collection_name=index_name)

if  collection_exists == False:
    create_collection = client_qdrant.create_collection(
        collection_name=index_name,
        vectors_config=models.VectorParams(size=dimensions, distance=models.Distance.COSINE),
    )
    
    print(f'A collection {index_name} foi criada!')
else:
    print(f'A collection {index_name} já existe!')

A collection PPGEGC_UFSC já existe!


### Upsert Points

In [8]:
id = 0
points = []

for index, row in df_qa_method.iterrows():
    id += 1
    autor = row.autor
    titulo = row.titulo
    tipo = row.tipo
    area_de_concentracao = row.area_de_concentracao
    ano_de_publicacao = row.ano_de_publicacao
    local = row.local
    orientador = row.orientador
    coorientador = row.coorientador
    resumo = row.resumo
    palavras_chave = row.palavras_chave
    abstract = row.abstract
    keywords = row.keywords
    introducao_contextualizacao = row.introducao_contextualizacao
    introducao_problematica = row.introducao_problematica
    introducao_ineditismo = row.introducao_ineditismo
    introducao_contribuicao = row.introducao_contribuicao
    conclusao = row.conclusao

    # Converta a string de embedding para uma lista de floats
    embedding = ast.literal_eval(row.embedding)

    points.append(models.PointStruct(id=id, vector=embedding,
                                payload={
                "autor": autor,
                "titulo": titulo,
                "tipo": tipo,
                "area_de_concentracao": area_de_concentracao,
                "ano_de_publicacao": ano_de_publicacao,
                "local": local,
                "orientador": orientador,
                "coorientador": coorientador,
                "resumo": resumo,
                "palavras_chave": palavras_chave,
                "abstract": abstract,
                "keywords": keywords, 
                "introducao_contextualizacao": introducao_contextualizacao,
                "introducao_problematica": introducao_problematica,
                "introducao_ineditismo": introducao_ineditismo,
                "introducao_contribuicao": introducao_contribuicao,
                "conclusao": conclusao}))

    if ((id % 50 ) == 0):
        operation_info = client_qdrant.upsert(
            collection_name=index_name,
            wait=True,
            points=points
        )
        points.clear()
        print(f"{id} tese/dissertação indexada!!!")

if len(points) != 0:
    operation_info = client_qdrant.upsert(
        collection_name=index_name,
        wait=True,
        points=points
    )
    print(f"{id} tese/dissertação indexada!")

print("Indexação terminada!")

50 tese/dissertação indexada!!!
100 tese/dissertação indexada!!!
150 tese/dissertação indexada!!!
Indexação terminada!
